# Classification models for enantioselectivity of the asymmetric Mannich reaction

Binary classification of reactions into high and low enantioselectivity, `ee ≥ threshold` (class 1) versus
`ee < threshold` (class 0), for thresholds of 70, 80 and 90% ee. Uses the representations computed in
`mannich_representations.ipynb` (and the precomputed `features_dft.csv`) and the
fold assignments of the regression notebook (`mannich_regression.ipynb`), so that regression and
classification are evaluated on identical test sets.

**Inputs** (set in the configuration cell)
- `mannich_dataset_hbd.csv` — reactions, `ee`, grouping columns, atom-mapped
  reaction SMILES, catalyst SMILES, axial configuration;
- `representations/features_*.csv` and `features_conditions.csv`;
- `folds_<split>.csv` from the results folder of the regression notebook (created here with the same
  procedure if absent).

**Models**: Dummy (class prior), logistic regression (L2), SVC (RBF), Gaussian process classifier,
Random Forest, XGBoost, chemprop v2 D-MPNN.

**Validation schemes**: random 5-fold × 5 repeats; leave-one-group-out by `research_group`;
GroupKFold (10 folds) by `catalyst_id`.

**Metrics**: ROC-AUC (primary), PR-AUC, MCC, F1, balanced accuracy; cluster-bootstrap 95% CIs.

**Outputs** (`results_classification/`): out-of-fold scores and labels, metric summary, class balance,
fold files, hyperparameters, heatmaps, ROC and precision–recall curves, `manifest.json`.

**Parallel runs**: thresholds, representations, splits, models, paths and thread counts can be overridden by
environment variables (`MANNICH_*`, see the configuration cell), so that independent parts run as separate
processes that share one fold-level cache. Worker runs set `MANNICH_COLLECT=0` and only fill the cache; a final
run with `MANNICH_RUN_MODELS=none` collects all cached results into tables and figures.

Sections: 1 Configuration · 2 Environment · 3 Data and labels · 4 Splits · 5 Models · 6 Classical models ·
7 chemprop · 8 Results · 9 Metrics · 10 Figures · 11 Manifest

## 1. Configuration
All paths and switches. Nothing below this cell needs editing.

In [ ]:
import os
from pathlib import Path

ENV = os.environ.get


def env_list(name, default, cast=str):
    # comma-separated override from the environment, e.g. MANNICH_THRESHOLDS=70,90
    value = ENV(name, "").strip()
    return [cast(v.strip()) for v in value.split(",") if v.strip()] if value else default


# --- Threads (must be set before numpy is imported) -----------------------------------
# One BLAS thread by default: multithreaded BLAS crashed the kernel on macOS (Apple M4).
# On Linux, parallel worker runs set MANNICH_BLAS_THREADS / MANNICH_N_JOBS to share the machine.
BLAS_THREADS = ENV("MANNICH_BLAS_THREADS", "1")
for _var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS_BLAS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_var, BLAS_THREADS)
N_JOBS = int(ENV("MANNICH_N_JOBS", "-1"))     # threads for RF / XGBoost (-1: all cores)
TORCH_THREADS = int(ENV("MANNICH_TORCH_THREADS", "0"))   # CPU threads of PyTorch (0: PyTorch default)

# --- Paths ---------------------------------------------------------------------------
# Default paths are relative to notebooks/ (run the notebook from that folder).
DATA_DIR = Path(ENV("MANNICH_DATA_DIR", "../data"))   # processed/ (dataset) and representations/
DATASET_FILE = "mannich_dataset_hbd.csv"
REPR_DIR = DATA_DIR / "representations"
FOLDS_DIR = Path(ENV("MANNICH_FOLDS_DIR", "../splits"))     # fold assignments shared with the regression notebook
RESULTS_DIR = Path(ENV("MANNICH_RESULTS_DIR", "../results/classification"))

# --- Target --------------------------------------------------------------------------
EE_COLUMN = "ee"                 # ee in %, absolute value is used
EE_THRESHOLDS = [70, 80, 90]     # class 1: ee >= threshold; each threshold is a separate task
DECISION_THRESHOLD = 0.5         # probability cut-off for class labels (SVC: decision function 0)

# --- What to run ---------------------------------------------------------------------
RUN_REPRESENTATIONS = ["physchem", "maccs", "ecfp4", "ecfp4_count", "drfp", "onehot", "dft"]
RUN_MODELS = ["dummy", "logreg", "svc", "gp", "rf", "xgb", "chemprop"]
RUN_SPLITS = ["random", "lorgo_group", "gkf10_catalyst"]
EXCLUDE_MODELS = []              # models left out of fitting and of the collected results, e.g. ["gp"]
COLLECT = True                   # sections 8-11: collect cached results, metrics, figures, manifest

# Overrides from the command line (used by the parallel run script):
#   MANNICH_RUN_MODELS=classical | chemprop | none     MANNICH_THRESHOLDS=70,80
#   MANNICH_REPRESENTATIONS=ecfp4,dft                  MANNICH_SPLITS=random
#   MANNICH_COLLECT=0 (worker run: fill the cache only)
#   MANNICH_EXCLUDE_MODELS=gp (leave models out of fitting and of the results; their cache is kept)
_run_models = ENV("MANNICH_RUN_MODELS", "").strip().lower()
if _run_models == "classical":
    RUN_MODELS = [m for m in RUN_MODELS if m != "chemprop"]
elif _run_models == "chemprop":
    RUN_MODELS = ["chemprop"]
elif _run_models == "none":
    RUN_MODELS = []
elif _run_models:
    raise ValueError(f"MANNICH_RUN_MODELS must be 'classical', 'chemprop' or 'none', got {_run_models!r}")
RUN_THRESHOLDS = env_list("MANNICH_THRESHOLDS", EE_THRESHOLDS, int)   # thresholds fitted in this run
RUN_REPRESENTATIONS = env_list("MANNICH_REPRESENTATIONS", RUN_REPRESENTATIONS)
RUN_SPLITS = env_list("MANNICH_SPLITS", RUN_SPLITS)
EXCLUDE_MODELS = env_list("MANNICH_EXCLUDE_MODELS", EXCLUDE_MODELS)
RUN_MODELS = [m for m in RUN_MODELS if m not in EXCLUDE_MODELS]
COLLECT = ENV("MANNICH_COLLECT", "1" if COLLECT else "0").strip() not in ("0", "false", "no")

# --- Cross-validation ----------------------------------------------------------------
SEED = 0                               # base seed (estimators, searches, bootstrap)
RANDOM_REPEAT_SEEDS = [0, 1, 2, 3, 4]  # KFold shuffling seeds (used only if fold files must be created)
N_FOLDS_RANDOM = 5
N_FOLDS_GKF_CATALYST = 10
N_INNER_FOLDS = 5                      # inner CV for hyperparameter search
N_ITER = 20                            # RandomizedSearchCV candidates per search
N_BOOT = 1000                          # cluster-bootstrap replicates
CI_LEVEL = 0.95

# --- Compute -------------------------------------------------------------------------
GP_N_RESTARTS = 2    # restarts of the marginal-likelihood optimiser

# --- chemprop (fixed hyperparameters, no nested search) -------------------------------
CHEMPROP_DEVICE = ENV("MANNICH_CHEMPROP_DEVICE", "auto")   # "cpu", "cuda", "mps" or "auto" (cuda > mps > cpu)
CHEMPROP = dict(
    message_hidden_dim=300, depth=3, dropout=0.1,
    ffn_hidden_dim=300, ffn_num_layers=2,
    batch_size=64, max_epochs=150, patience=20,
    init_lr=1e-4, max_lr=1e-3, final_lr=1e-4, warmup_epochs=2,
    val_fraction=0.1,      # internal validation part for early stopping
)
print(f"thresholds fitted: {RUN_THRESHOLDS} | representations: {RUN_REPRESENTATIONS} | splits: {RUN_SPLITS} | "
      f"models: {RUN_MODELS} | excluded: {EXCLUDE_MODELS} | collect: {COLLECT} | BLAS threads: {BLAS_THREADS} | N_JOBS: {N_JOBS}")

## 2. Environment check

In [ ]:
import platform
import sys
from importlib import metadata

DISTRIBUTIONS = {"sklearn": "scikit-learn"}


def package_version(name):
    """Installed version from package metadata; the package itself is not imported."""
    try:
        return metadata.version(DISTRIBUTIONS.get(name, name))
    except metadata.PackageNotFoundError:
        return None


PACKAGES = ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "xgboost",
            "torch", "lightning", "chemprop", "rdkit"]
VERSIONS = {p: package_version(p) for p in PACKAGES}

print(f"Python {sys.version.split()[0]} | {platform.platform()}")
for p, v in VERSIONS.items():
    print(f"  {p:<11} {v or 'not installed'}")

required = {"numpy", "pandas", "scipy", "sklearn", "matplotlib"}
if "xgb" in RUN_MODELS:
    required.add("xgboost")
if "chemprop" in RUN_MODELS:
    required |= {"chemprop", "torch", "lightning", "rdkit"}
missing = sorted(p for p in required if VERSIONS[p] is None)
if missing:
    raise ImportError(f"Missing packages for the current RUN_MODELS: {missing}")

if "chemprop" in RUN_MODELS:
    if int(VERSIONS["chemprop"].split(".")[0]) < 2:
        raise ImportError("chemprop >= 2.0 is required")
    import torch
    print(f"  torch backends: MPS={torch.backends.mps.is_available()}, CUDA={torch.cuda.is_available()}")
if "chemprop" in RUN_MODELS and len(RUN_MODELS) > 1:
    print("  note: chemprop (PyTorch) and the classical models (XGBoost) share one process in this run; "
          "on macOS run them separately (MANNICH_RUN_MODELS=classical / chemprop) to avoid OpenMP conflicts")

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # CPU fallback for ops missing on MPS

import gc
import hashlib
import json
import logging
import pickle
import shutil
import time
import warnings
from dataclasses import replace
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import TwoSlopeNorm
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import (ConstantKernel, Hyperparameter, Kernel, Matern,
                                              NormalizedKernelMixin, StationaryKernelMixin)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, f1_score,
                             matthews_corrcoef, precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit, KFold, RandomizedSearchCV,
                                     StratifiedGroupKFold, StratifiedKFold, StratifiedShuffleSplit)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

if "xgb" in RUN_MODELS:
    from xgboost import XGBClassifier

CODE_VERSION = "1"   # bump to invalidate cached fold results after code changes
CACHE_DIR = RESULTS_DIR / "cache"
FIG_DIR = RESULTS_DIR / "figures"
for d in (RESULTS_DIR, CACHE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
LOG_FILE = RESULTS_DIR / "run_log.txt"


def memory_mb():
    """Resident memory of this process (MB); peak RSS if psutil is unavailable."""
    try:
        import psutil
        return psutil.Process().memory_info().rss / 2**20
    except Exception:
        import resource
        peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return peak / 2**20 if sys.platform == "darwin" else peak / 2**10


def log(message):
    """Print and append a time-stamped line to run_log.txt (progress survives a lost front end)."""
    print(message, flush=True)
    with open(LOG_FILE, "a") as fh:
        fh.write(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  {message}\n")


def release_memory():
    """Free memory between representations: GP kernel cache, joblib worker processes, garbage."""
    try:
        _SimilarityKernel._CACHE.clear()
    except NameError:
        pass
    try:
        from joblib.externals.loky import get_reusable_executor
        get_reusable_executor().shutdown(wait=True)
    except Exception:
        pass
    gc.collect()


import faulthandler
_fault_log = open(RESULTS_DIR / "faulthandler.log", "a")
_fault_log.write(f"\n=== session {datetime.now().isoformat(timespec='seconds')}\n")
_fault_log.flush()
faulthandler.enable(file=_fault_log, all_threads=True)   # Python traceback of a hard crash goes here

try:   # enforce the BLAS thread limit at run time as well (environment variables may be read too late)
    from threadpoolctl import threadpool_limits
    _blas_limit = threadpool_limits(limits=int(BLAS_THREADS), user_api="blas")
except Exception:
    pass

log(f"=== session start | pid {os.getpid()} | RUN_MODELS={RUN_MODELS} | thresholds={RUN_THRESHOLDS} | "
    f"representations={RUN_REPRESENTATIONS} | splits={RUN_SPLITS} | "
    f"torch loaded: {'torch' in sys.modules} | xgboost loaded: {'xgboost' in sys.modules} | "
    f"memory {memory_mb():.0f} MB")

plt.rcParams.update({"font.size": 8, "pdf.fonttype": 42, "ps.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})

## 3. Data and labels

Class labels are taken from `ee` (absolute value, %) of each reaction: class 1 if `ee ≥ threshold`. A fixed
ee threshold corresponds to different |ΔΔG‡| at different reaction temperatures, so labels are defined on ee
rather than on `ddG_kcal_mol`. Each threshold is treated as a separate classification task with its own
hyperparameter search.

Rows of every representation file must carry the same `reaction_id` values in the same order as the
dataset; a mismatch stops the notebook. Each representation is merged with `features_conditions.csv`
on `reaction_id`.

Representation types define fold-internal preprocessing and the GP kernel:
- *continuous* (physchem, DFT): all columns standardised; GP with Matérn (ν = 2.5) kernel;
- *binary* (MACCS, ECFP4, DRFP, one-hot) and *count* (ECFP4-count): the fingerprint block is not scaled;
  conditions are standardised; GP with Tanimoto (binary) or MinMax (count) kernel on the fingerprint block
  plus an RBF kernel on the conditions.

In [ ]:
REPRESENTATIONS = {   # name: (file, type)
    "physchem":    ("features_physchem.csv", "continuous"),
    "maccs":       ("features_maccs.csv", "binary"),
    "ecfp4":       ("features_ecfp4.csv", "binary"),
    "ecfp4_count": ("features_ecfp4_count.csv", "count"),
    "drfp":        ("features_drfp.csv", "binary"),
    "onehot":      ("features_onehot.csv", "binary"),
    "dft":         ("features_dft.csv", "continuous"),
}
OPTIONAL_REPRESENTATIONS = {"dft"}
CONDITIONS_FILE = "features_conditions.csv"

REP_LABELS = {"physchem": "Physchem", "maccs": "MACCS", "ecfp4": "ECFP4", "ecfp4_count": "ECFP4-count",
              "drfp": "DRFP", "onehot": "OHE", "dft": "DFT", "cgr": "CGR + catalyst"}
MODEL_LABELS = {"dummy": "Dummy", "logreg": "LogReg", "svc": "SVC", "gp": "GPC", "rf": "RF",
                "xgb": "XGBoost", "chemprop": "D-MPNN"}
REP_ORDER = list(REPRESENTATIONS) + ["cgr"]
MODEL_ORDER = list(MODEL_LABELS)

unknown = (set(RUN_REPRESENTATIONS) - set(REPRESENTATIONS)) | (set(RUN_MODELS) - set(MODEL_LABELS))
if unknown:
    raise ValueError(f"Unknown names in RUN_REPRESENTATIONS / RUN_MODELS: {sorted(unknown)}")
if not EE_THRESHOLDS or any(not 0 < t < 100 for t in EE_THRESHOLDS) or len(set(EE_THRESHOLDS)) != len(EE_THRESHOLDS):
    raise ValueError(f"EE_THRESHOLDS must be distinct values in (0, 100), got {EE_THRESHOLDS}")
if set(RUN_THRESHOLDS) - set(EE_THRESHOLDS):
    raise ValueError(f"MANNICH_THRESHOLDS must be a subset of EE_THRESHOLDS {EE_THRESHOLDS}, got {RUN_THRESHOLDS}")


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


# --- dataset -------------------------------------------------------------------------
DATASET_PATH = DATA_DIR / "processed" / DATASET_FILE
REQUIRED_COLUMNS = ["reaction_id", EE_COLUMN, "research_group", "catalyst_id", "paper_doi",
                    "mapped_rxn", "organocatalyst", "axial_configuration"]
data = pd.read_csv(DATASET_PATH)
missing_cols = [c for c in REQUIRED_COLUMNS if c not in data.columns]
if missing_cols:
    raise ValueError(f"Dataset lacks columns: {missing_cols}")
if data["reaction_id"].duplicated().any():
    raise ValueError("Duplicated reaction_id in the dataset")
for c in [EE_COLUMN, "research_group", "catalyst_id"]:
    if data[c].isna().any():
        raise ValueError(f"Missing values in {c}")

REACTION_IDS = data["reaction_id"].to_numpy()
POS = pd.Series(np.arange(len(data)), index=REACTION_IDS)   # reaction_id -> row position
EE_RAW = data[EE_COLUMN].to_numpy(float)
EE = np.abs(EE_RAW)
if EE.max() > 100:
    raise ValueError(f"{EE_COLUMN} exceeds 100 (expected %)")
if EE.max() <= 1:
    raise ValueError(f"{EE_COLUMN} looks like a fraction; percent is expected")
GROUPS = {"research_group": data["research_group"].astype(str).to_numpy(),
          "catalyst_id": data["catalyst_id"].astype(str).to_numpy()}
LABELS = {t: (EE >= t).astype(int) for t in EE_THRESHOLDS}
LABEL_HASHES = {t: hashlib.sha256(LABELS[t].tobytes()).hexdigest()[:16] for t in EE_THRESHOLDS}

axial = data["axial_configuration"].fillna("none").astype(str).str.strip()
axial = axial.map(lambda s: {"r": "R", "s": "S", "none": "none"}.get(s.lower(), s))
bad_axial = sorted(set(axial) - {"R", "S", "none"})
if bad_axial:
    raise ValueError(f"Unexpected axial_configuration values: {bad_axial}")
AXIAL_FLAGS = np.column_stack([axial == "R", axial == "S"]).astype(float)

cat_counts = data["catalyst_id"].value_counts()
print(f"Reactions: {len(data)}")
print(f"research_group: {data['research_group'].nunique()} | paper_doi: {data['paper_doi'].nunique()} | "
      f"catalyst_id: {len(cat_counts)} ({(cat_counts == 1).sum()} with a single reaction)")
print(f"{EE_COLUMN} (%): mean {EE.mean():.1f}, median {np.median(EE):.1f}, min {EE.min():.1f}, max {EE.max():.1f}"
      f"{f'; {int((EE_RAW < 0).sum())} negative values used as absolute' if (EE_RAW < 0).any() else ''}")
for t in EE_THRESHOLDS:
    print(f"  ee >= {t}%: class 1 = {LABELS[t].sum()} / {len(EE)} (prevalence {LABELS[t].mean():.3f})")


def check_reaction_ids(frame, label):
    """Same reaction_id values, in the same order, as the dataset."""
    if frame.columns[0] != "reaction_id":
        raise ValueError(f"{label}: first column must be 'reaction_id', found {frame.columns[0]!r}")
    ids = frame["reaction_id"].to_numpy()
    if len(ids) != len(REACTION_IDS):
        raise ValueError(f"{label}: {len(ids)} rows, dataset has {len(REACTION_IDS)}")
    if not np.array_equal(ids, REACTION_IDS):
        what = "order" if set(ids) == set(REACTION_IDS) else "set"
        raise ValueError(f"{label}: reaction_id {what} differs from the dataset")


# --- conditions ----------------------------------------------------------------------
conditions = pd.read_csv(REPR_DIR / CONDITIONS_FILE)
check_reaction_ids(conditions, CONDITIONS_FILE)
COND_COLS = [c for c in conditions.columns if c != "reaction_id"]
print(f"Conditions ({len(COND_COLS)}): {COND_COLS}; "
      f"missing values: {int(conditions[COND_COLS].isna().sum().sum())}")

# --- input hashes (manifest, cache signatures) ---------------------------------------
FILE_HASHES = {DATASET_FILE: sha256(DATASET_PATH), CONDITIONS_FILE: sha256(REPR_DIR / CONDITIONS_FILE)}
for name, (fname, _) in REPRESENTATIONS.items():
    if (REPR_DIR / fname).exists():
        FILE_HASHES[fname] = sha256(REPR_DIR / fname)
    elif name not in OPTIONAL_REPRESENTATIONS and name in RUN_REPRESENTATIONS:
        raise FileNotFoundError(REPR_DIR / fname)
print("Representation files found:",
      [n for n, (f, _) in REPRESENTATIONS.items() if f in FILE_HASHES])

In [ ]:
def load_representation(name):
    """Representation block merged with conditions. Returns None if an optional file is absent.

    X columns are ordered [representation | conditions]; n_fp is the size of the fingerprint
    block (0 for continuous representations, where all columns are treated as continuous).
    """
    fname, kind = REPRESENTATIONS[name]
    path = REPR_DIR / fname
    if not path.exists():
        if name in OPTIONAL_REPRESENTATIONS:
            return None
        raise FileNotFoundError(path)
    rep = pd.read_csv(path)
    check_reaction_ids(rep, fname)
    rep_cols = [c for c in rep.columns if c != "reaction_id"]
    overlap = set(rep_cols) & set(COND_COLS)
    if overlap:
        print(f"  {fname}: columns also in conditions, kept once from conditions: {sorted(overlap)}")
        rep_cols = [c for c in rep_cols if c not in overlap]
    non_numeric = [c for c in rep_cols if not pd.api.types.is_numeric_dtype(rep[c])]
    if non_numeric:
        raise ValueError(f"{fname}: non-numeric columns {non_numeric[:5]}")

    merged = rep[["reaction_id"] + rep_cols].merge(conditions, on="reaction_id", how="inner",
                                                    validate="one_to_one")
    check_reaction_ids(merged, f"{fname} + {CONDITIONS_FILE}")

    Xr = merged[rep_cols].to_numpy(float)
    values = Xr[~np.isnan(Xr)]
    if kind == "binary" and not np.isin(values, (0.0, 1.0)).all():
        raise ValueError(f"{fname}: declared binary but contains values other than 0/1")
    if kind == "count" and ((values < 0).any() or (values != np.round(values)).any()):
        raise ValueError(f"{fname}: declared count but contains negative or non-integer values")

    if {"cat_axial_R", "cat_axial_S"} <= set(rep_cols):
        flags = merged[["cat_axial_R", "cat_axial_S"]].to_numpy(float)
        if not np.array_equal(flags, AXIAL_FLAGS):
            raise ValueError(f"{fname}: cat_axial_R/S disagree with axial_configuration")

    X = np.hstack([Xr, merged[COND_COLS].to_numpy(float)])
    n_fp = 0 if kind == "continuous" else len(rep_cols)
    return {"name": name, "kind": kind, "X": X, "n_fp": n_fp, "columns": rep_cols + COND_COLS,
            "file": fname}

## 4. Splits

| Split | Outer folds | Inner folds (search) | Bootstrap cluster |
|---|---|---|---|
| `random` | KFold 5 × 5 repeats (seeds 0–4) | StratifiedKFold 5 | reaction |
| `lorgo_group` | leave-one-group-out, `research_group` (43 folds) | StratifiedGroupKFold 5, `research_group` | `research_group` |
| `gkf10_catalyst` | GroupKFold 10, `catalyst_id` | StratifiedGroupKFold 5, `catalyst_id` | `catalyst_id` |

The outer folds are read from `folds_<split>.csv` of the regression notebook (copied to the results
folder of this notebook), so regression and classification share their test sets. The outer folds are not
stratified by class; the class balance of the test folds is reported below. Under leave-one-group-out, a test
fold can contain one class only; metrics are therefore computed on pooled out-of-fold predictions (Section 9),
never per fold. Inner folds of the hyperparameter search are stratified by class (and grouped as the outer
split), so that every inner validation fold contains both classes.

In [ ]:
ALL_SPLITS = ["random", "lorgo_group", "gkf10_catalyst"]
SPLIT_GROUP = {"random": None, "lorgo_group": "research_group", "gkf10_catalyst": "catalyst_id"}
SPLIT_LABELS = {"random": "Random (5-fold × 5)", "lorgo_group": "LORGO (research group)",
                "gkf10_catalyst": "GroupKFold-10 (catalyst)"}
if set(RUN_SPLITS) - set(ALL_SPLITS):
    raise ValueError(f"Unknown splits: {sorted(set(RUN_SPLITS) - set(ALL_SPLITS))}")


def make_folds(split):
    """Same procedure as in the regression notebook (used only if its fold files are absent)."""
    n = len(REACTION_IDS)
    if split == "random":
        parts = []
        for repeat, seed in enumerate(RANDOM_REPEAT_SEEDS):
            fold = np.empty(n, dtype=int)
            for k, (_, te) in enumerate(KFold(N_FOLDS_RANDOM, shuffle=True, random_state=seed).split(np.zeros(n))):
                fold[te] = k
            parts.append(pd.DataFrame({"reaction_id": REACTION_IDS, "repeat": repeat, "fold": fold,
                                       "group": ""}))
        return pd.concat(parts, ignore_index=True)
    g = GROUPS[SPLIT_GROUP[split]]
    if split == "lorgo_group":
        fold = np.searchsorted(np.unique(g), g)
    else:
        fold = np.empty(n, dtype=int)
        for k, (_, te) in enumerate(GroupKFold(N_FOLDS_GKF_CATALYST).split(np.zeros(n), groups=g)):
            fold[te] = k
    return pd.DataFrame({"reaction_id": REACTION_IDS, "repeat": 0, "fold": fold, "group": g})


def validate_folds(folds, split):
    for repeat, sub in folds.groupby("repeat"):
        if len(sub) != len(REACTION_IDS) or set(sub["reaction_id"]) != set(REACTION_IDS):
            raise ValueError(f"folds_{split}.csv, repeat {repeat}: does not cover each reaction once")
    if split != "random":
        g = pd.Series(GROUPS[SPLIT_GROUP[split]], index=REACTION_IDS)
        if not (folds["group"].astype(str).to_numpy() == g.loc[folds["reaction_id"]].to_numpy()).all():
            raise ValueError(f"folds_{split}.csv: group labels differ from the dataset")
        if (folds.groupby("group")["fold"].nunique() > 1).any():
            raise ValueError(f"folds_{split}.csv: a group is spread over several folds")


FOLDS, FOLD_HASHES = {}, {}
for split in ALL_SPLITS:
    source = FOLDS_DIR / f"folds_{split}.csv"
    target = FOLDS_DIR / f"folds_{split}.csv"
    if source.exists():
        folds = pd.read_csv(source, dtype={"group": str}, keep_default_na=False)
        validate_folds(folds, split)
        if source.resolve() != target.resolve() and (not target.exists() or sha256(target) != sha256(source)):
            tmp = target.with_suffix(f".{os.getpid()}.tmp")   # atomic: parallel runs may copy at the same time
            shutil.copyfile(source, tmp)
            os.replace(tmp, target)
        status = "read from splits/"
    elif target.exists():
        folds = pd.read_csv(target, dtype={"group": str}, keep_default_na=False)
        validate_folds(folds, split)
        status = "read from this results folder"
    else:
        folds = make_folds(split)
        folds.insert(1, "split", split)
        validate_folds(folds, split)
        tmp = target.with_suffix(f".{os.getpid()}.tmp")
        folds.to_csv(tmp, index=False)
        os.replace(tmp, target)
        status = "created (regression fold file not found)"
    FOLDS[split] = folds
    FOLD_HASHES[split] = sha256(source if source.exists() else target)
    sizes = folds.groupby(["repeat", "fold"]).size()
    print(f"{split:<15} repeats={folds['repeat'].nunique()} folds/repeat={folds['fold'].nunique():>3} "
          f"test size min/median/max = {sizes.min()}/{int(sizes.median())}/{sizes.max()} | {status}")


def iter_folds(split):
    """Yields (repeat, fold, train_positions, test_positions)."""
    folds = FOLDS[split]
    for repeat, rep_folds in folds.groupby("repeat", sort=True):
        for fold in sorted(rep_folds["fold"].unique()):
            is_test = (rep_folds["fold"] == fold).to_numpy()
            ids = rep_folds["reaction_id"].to_numpy()
            yield int(repeat), int(fold), POS.loc[ids[~is_test]].to_numpy(), POS.loc[ids[is_test]].to_numpy()


def n_fold_units(split):
    return FOLDS[split].groupby(["repeat", "fold"]).ngroups

In [ ]:
# class balance of the outer test folds
balance_rows = []
for t in EE_THRESHOLDS:
    y = LABELS[t]
    for split in ALL_SPLITS:
        prev = []
        for _, _, train, test in iter_folds(split):
            prev.append(y[test].mean())
            if y[train].min() == y[train].max():
                raise ValueError(f"ee >= {t}%, {split}: a training fold contains one class only")
        prev = np.asarray(prev)
        balance_rows.append({"threshold": t, "split": split, "prevalence": y.mean(), "n_test_folds": len(prev),
                             "test_prevalence_min": prev.min(), "test_prevalence_median": np.median(prev),
                             "test_prevalence_max": prev.max(),
                             "test_folds_class0_only": int((prev == 0).sum()),
                             "test_folds_class1_only": int((prev == 1).sum())})
CLASS_BALANCE = pd.DataFrame(balance_rows)
if COLLECT:
    CLASS_BALANCE.to_csv(RESULTS_DIR / "class_balance.csv", index=False, float_format="%.4f")
display(CLASS_BALANCE.style.hide(axis="index").format(precision=3))

## 5. Models

**Fold-internal preprocessing** (`BlockPreprocessor`, fitted on training rows only): median imputation,
removal of columns constant in the training rows, standardisation of continuous columns (logistic regression,
SVC, GPC; not for trees). For one-hot encoding, a component absent from the training rows gives an all-zero
column that is removed, so a test reaction with that component is encoded by a zero block for it.

**Class imbalance**: class weights inversely proportional to class frequencies in the training part
(`class_weight="balanced"` for logistic regression, SVC and RF; `scale_pos_weight = n₀/n₁` for XGBoost;
per-reaction loss weights for chemprop). The Gaussian process classifier has no class weights. No resampling.

**Scores and labels**: the score of a reaction is the predicted probability of class 1 (SVC: decision
function value); it is used for ROC-AUC and PR-AUC. Class labels for MCC, F1 and balanced accuracy are
`probability ≥ 0.5` (SVC: decision function ≥ 0); the cut-off is not tuned. The Dummy baseline has a constant
score (ROC-AUC = 0.5, PR-AUC = prevalence) and predicts the majority class of the training part.

**Hyperparameters** (logistic regression, SVC, RF, XGBoost): `RandomizedSearchCV` with `N_ITER` candidates,
ROC-AUC criterion, one search per threshold × split scheme × representation × model on the training part of the
first outer fold (repeat 0, fold 0); the selected values are then fixed and each outer fold refits the model
with them. The test reactions of the first fold never enter the search; reactions in the test parts of the other
folds take part in choosing the hyperparameters (not in fitting the evaluated models), as in the regression
notebook. GPC kernel hyperparameters are fitted in every fold by maximising the Laplace approximation of the
log marginal likelihood; chemprop uses fixed hyperparameters (Section 7).

**GPC kernels** (no white-noise term; the Bernoulli likelihood carries the label noise)
- fingerprints: `C·Tanimoto(fp) + C·RBF_ARD(conditions)`; MinMax instead of Tanimoto for ECFP4-count;
- continuous: `C·Matérn(ν=2.5)` on standardised features.

In [ ]:
class BlockPreprocessor(BaseEstimator, TransformerMixin):
    """Imputation, constant-column removal and optional scaling for X = [fingerprint | continuous].

    All statistics come from the rows passed to fit(). The output keeps the block order;
    `n_fp_out_` is the number of retained fingerprint columns.
    """

    def __init__(self, n_fp=0, scale_continuous=True):
        self.n_fp = n_fp
        self.scale_continuous = scale_continuous

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)   # all-NaN columns
            med = np.nanmedian(X, axis=0)
        self.medians_ = np.where(np.isnan(med), 0.0, med)
        Xi = np.where(np.isnan(X), self.medians_, X)
        keep = Xi.max(axis=0) > Xi.min(axis=0)
        is_fp = np.arange(X.shape[1]) < self.n_fp
        self.fp_idx_ = np.flatnonzero(keep & is_fp)
        self.cont_idx_ = np.flatnonzero(keep & ~is_fp)
        self.n_fp_out_ = len(self.fp_idx_)
        if self.scale_continuous and len(self.cont_idx_):
            C = Xi[:, self.cont_idx_]
            self.mean_ = C.mean(axis=0)
            sd = C.std(axis=0)
            self.scale_ = np.where(sd > 0, sd, 1.0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        Xi = np.where(np.isnan(X), self.medians_, X)
        cont = Xi[:, self.cont_idx_]
        if self.scale_continuous and len(self.cont_idx_):
            cont = (cont - self.mean_) / self.scale_
        return np.hstack([Xi[:, self.fp_idx_], cont])


# --- GP kernels ----------------------------------------------------------------------
class _SimilarityKernel(Kernel):
    """Similarity on columns [start, stop) of X; no hyperparameters.

    K(X, X) is cached (keyed by content hash), because sklearn re-evaluates the kernel at every
    step of the marginal-likelihood optimisation while only the other kernel terms change.
    """
    _CACHE = {}

    def __init__(self, start=0, stop=None):
        self.start = start
        self.stop = stop

    def _block(self, X):
        return np.asarray(X, dtype=float)[:, self.start:self.stop]

    def _similarity(self, A, B):
        raise NotImplementedError

    def __call__(self, X, Y=None, eval_gradient=False):
        A = self._block(X)
        if Y is None:
            key = (type(self).__name__, hashlib.blake2b(np.ascontiguousarray(A).tobytes(), digest_size=16).hexdigest())
            if key not in self._CACHE:
                if len(self._CACHE) >= 4:
                    self._CACHE.clear()
                self._CACHE[key] = self._similarity(A, A)
            K = self._CACHE[key].copy()
        else:
            if eval_gradient:
                raise ValueError("Gradient can only be evaluated when Y is None.")
            K = self._similarity(A, self._block(Y))
        if eval_gradient:
            return K, np.empty((K.shape[0], K.shape[0], 0))
        return K

    def diag(self, X):
        return np.ones(np.asarray(X).shape[0])

    def is_stationary(self):
        return False

    def __repr__(self):
        return f"{type(self).__name__}(cols={self.start}:{self.stop})"


class TanimotoKernel(_SimilarityKernel):
    """<a,b> / (|a|² + |b|² − <a,b>); equals Jaccard similarity for binary vectors."""

    def _similarity(self, A, B):
        AB = (A.astype(np.float32) @ B.astype(np.float32).T).astype(np.float64)
        denom = (A * A).sum(1)[:, None] + (B * B).sum(1)[None, :] - AB
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where(denom > 0, AB / denom, 1.0)


class MinMaxKernel(_SimilarityKernel):
    """Σ min(a,b) / Σ max(a,b) for non-negative integer counts.

    Σ min is computed as Σ_t <[a ≥ t], [b ≥ t]> over count thresholds t (matrix products).
    """

    def _similarity(self, A, B):
        A = np.rint(A).astype(np.int32)
        B = np.rint(B).astype(np.int32)
        col_max = np.maximum(A.max(0, initial=0), B.max(0, initial=0))
        smin = np.zeros((A.shape[0], B.shape[0]))
        for t in range(1, int(col_max.max(initial=0)) + 1):
            cols = np.flatnonzero(col_max >= t)
            smin += ((A[:, cols] >= t).astype(np.float32) @ (B[:, cols] >= t).astype(np.float32).T)
        smax = A.sum(1)[:, None] + B.sum(1)[None, :] - smin
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where(smax > 0, smin / smax, 1.0)


class SlicedRBF(StationaryKernelMixin, NormalizedKernelMixin, Kernel):
    """RBF kernel (scalar or ARD length scales) on columns [start, stop) of X."""

    def __init__(self, start=0, stop=None, length_scale=1.0, length_scale_bounds=(1e-2, 1e3)):
        self.start = start
        self.stop = stop
        self.length_scale = length_scale
        self.length_scale_bounds = length_scale_bounds

    @property
    def anisotropic(self):
        return np.iterable(self.length_scale) and len(self.length_scale) > 1

    @property
    def hyperparameter_length_scale(self):
        if self.anisotropic:
            return Hyperparameter("length_scale", "numeric", self.length_scale_bounds, len(self.length_scale))
        return Hyperparameter("length_scale", "numeric", self.length_scale_bounds)

    def __call__(self, X, Y=None, eval_gradient=False):
        A = np.asarray(X, dtype=float)[:, self.start:self.stop]
        ls = np.squeeze(np.asarray(self.length_scale, dtype=float))
        if Y is None:
            D = squareform(pdist(A / ls, metric="sqeuclidean"))
        else:
            if eval_gradient:
                raise ValueError("Gradient can only be evaluated when Y is None.")
            D = cdist(A / ls, np.asarray(Y, dtype=float)[:, self.start:self.stop] / ls, metric="sqeuclidean")
        K = np.exp(-0.5 * D)
        if not eval_gradient:
            return K
        if self.hyperparameter_length_scale.fixed:
            return K, np.empty((K.shape[0], K.shape[0], 0))
        if self.anisotropic:
            G = (A[:, None, :] - A[None, :, :]) ** 2 / ls ** 2
            return K, G * K[..., None]
        return K, (K * D)[..., None]

    def __repr__(self):
        ls = ", ".join(f"{v:.3g}" for v in np.atleast_1d(self.length_scale))
        return f"RBF(cols={self.start}:{self.stop}, length_scale=[{ls}])"


class GPClassifierBlocks(ClassifierMixin, BaseEstimator):
    """Gaussian process classifier (Laplace) with fold-internal preprocessing and a kernel by representation type."""

    def __init__(self, kind="binary", n_fp=0, n_restarts=2, random_state=0):
        self.kind = kind
        self.n_fp = n_fp
        self.n_restarts = n_restarts
        self.random_state = random_state

    def _kernel(self, n_fp, d):
        amp = lambda: ConstantKernel(1.0, (1e-3, 1e3))
        if self.kind == "continuous" or n_fp == 0:
            return amp() * Matern(length_scale=np.sqrt(d), length_scale_bounds=(1e-2, 1e4), nu=2.5)
        sim = TanimotoKernel(0, n_fp) if self.kind == "binary" else MinMaxKernel(0, n_fp)
        kernel = amp() * sim
        if d > n_fp:
            kernel = kernel + amp() * SlicedRBF(n_fp, d, length_scale=np.ones(d - n_fp))
        return kernel

    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        self.prep_ = BlockPreprocessor(self.n_fp, scale_continuous=True).fit(X)
        Z = self.prep_.transform(X)
        self.gpc_ = GaussianProcessClassifier(self._kernel(self.prep_.n_fp_out_, Z.shape[1]),
                                              n_restarts_optimizer=self.n_restarts,
                                              random_state=self.random_state)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            self.gpc_.fit(Z, np.asarray(y).astype(int))
        return self

    def predict_proba(self, X):
        return self.gpc_.predict_proba(self.prep_.transform(X))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= DECISION_THRESHOLD).astype(int)

    def fitted_params(self):
        return {"kernel": str(self.gpc_.kernel_),
                "log_marginal_likelihood": float(self.gpc_.log_marginal_likelihood_value_)}


class BalancedXGBClassifier(ClassifierMixin, BaseEstimator):
    """XGBoost classifier with scale_pos_weight = n0 / n1 computed on the training rows."""

    def __init__(self, max_depth=6, learning_rate=0.1, n_estimators=300, subsample=1.0,
                 colsample_bytree=1.0, reg_lambda=1.0, random_state=0, n_jobs=1):
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.reg_lambda = reg_lambda
        self.random_state = random_state
        self.n_jobs = n_jobs

    def fit(self, X, y):
        y = np.asarray(y).astype(int)
        self.classes_ = np.array([0, 1])
        n1 = y.sum()
        self.scale_pos_weight_ = (len(y) - n1) / max(n1, 1)
        self.model_ = XGBClassifier(max_depth=self.max_depth, learning_rate=self.learning_rate,
                                    n_estimators=self.n_estimators, subsample=self.subsample,
                                    colsample_bytree=self.colsample_bytree, reg_lambda=self.reg_lambda,
                                    scale_pos_weight=self.scale_pos_weight_, objective="binary:logistic",
                                    eval_metric="logloss", tree_method="hist", random_state=self.random_state,
                                    n_jobs=self.n_jobs, verbosity=0)
        self.model_.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= DECISION_THRESHOLD).astype(int)

In [ ]:
# Search spaces: declarative (for cache signatures), converted to scipy distributions below.
PARAM_SPACES = {
    "logreg": {"model__C": ("loguniform", 1e-4, 1e2)},
    "svc": {"model__C": ("loguniform", 1e-2, 1e3),
            "model__gamma": ("loguniform", 1e-5, 1e-1)},
    "rf": {"model__n_estimators": ("choice", [200, 500]),
           "model__max_depth": ("choice", [None, 8, 16, 32]),
           "model__max_features": ("choice", ["sqrt", 0.1, 0.3]),
           "model__min_samples_leaf": ("choice", [1, 2, 4, 8])},
    "xgb": {"model__max_depth": ("randint", 2, 8),
            "model__learning_rate": ("loguniform", 0.01, 0.3),
            "model__n_estimators": ("randint", 100, 1000),
            "model__subsample": ("uniform", 0.6, 1.0),
            "model__colsample_bytree": ("uniform", 0.3, 1.0),
            "model__reg_lambda": ("loguniform", 1e-2, 1e2)},
}


def to_distribution(spec):
    kind, *args = spec
    if kind == "loguniform":
        return loguniform(args[0], args[1])
    if kind == "uniform":                      # [low, high]
        return uniform(args[0], args[1] - args[0])
    if kind == "randint":                      # [low, high], inclusive
        return randint(args[0], args[1] + 1)
    if kind == "choice":
        return list(args[0])
    raise ValueError(kind)


def make_estimator(model, rep, seed):
    n_fp = rep["n_fp"]
    if model == "dummy":
        return DummyClassifier(strategy="prior")
    if model == "logreg":
        return Pipeline([("prep", BlockPreprocessor(n_fp, True)),
                         ("model", LogisticRegression(class_weight="balanced", max_iter=5000))])
    if model == "svc":
        return Pipeline([("prep", BlockPreprocessor(n_fp, True)),
                         ("model", SVC(kernel="rbf", class_weight="balanced"))])
    if model == "gp":
        return GPClassifierBlocks(kind=rep["kind"], n_fp=n_fp, n_restarts=GP_N_RESTARTS, random_state=seed)
    if model == "rf":
        return Pipeline([("prep", BlockPreprocessor(n_fp, False)),
                         ("model", RandomForestClassifier(class_weight="balanced", random_state=seed,
                                                          n_jobs=N_JOBS))])
    if model == "xgb":
        return Pipeline([("prep", BlockPreprocessor(n_fp, False)),
                         ("model", BalancedXGBClassifier(random_state=seed, n_jobs=N_JOBS))])
    raise ValueError(model)


def scores_and_labels(model, est, X):
    """Score (probability of class 1; SVC: decision function) and class label at the fixed cut-off.

    Dummy: constant score 0 in every fold (no ranking information) and the majority class of the training
    part as label. The class prior is not used as score, because it differs slightly between folds and the
    pooled ranking would then follow the fold composition (ROC-AUC away from 0.5 without any model).
    """
    if model == "dummy":
        return np.zeros(len(X)), est.predict(X).astype(int)
    if model == "svc":
        score = est.decision_function(X)
        return score, (score >= 0).astype(int)
    score = est.predict_proba(X)[:, 1]
    return score, (score >= DECISION_THRESHOLD).astype(int)


def inner_cv(split, n_train_groups, seed):
    if split == "random":
        return StratifiedKFold(N_INNER_FOLDS, shuffle=True, random_state=seed)
    return StratifiedGroupKFold(min(N_INNER_FOLDS, n_train_groups))


def to_builtin(obj):
    """JSON-serialisable copy of a parameter dict."""
    if isinstance(obj, dict):
        return {k: to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


def run_search(model, rep, split, threshold, train, seed):
    """RandomizedSearchCV on the rows `train`; returns (best params, inner ROC-AUC)."""
    est = make_estimator(model, rep, seed)
    groups = None if split == "random" else GROUPS[SPLIT_GROUP[split]][train]
    n_groups = len(np.unique(groups)) if groups is not None else len(train)
    space = {k: to_distribution(v) for k, v in PARAM_SPACES[model].items()}
    search = RandomizedSearchCV(est, space, n_iter=N_ITER, scoring="roc_auc",
                                cv=inner_cv(split, n_groups, seed), refit=False, random_state=seed,
                                n_jobs=1, error_score="raise")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        search.fit(rep["X"][train], LABELS[threshold][train], groups=groups)
    return search.best_params_, float(search.best_score_)


def tuned_once(model, rep, split, threshold):
    """Search on the training part of the first outer fold (repeat 0, fold 0), cached on disk."""
    path = combo_dir(threshold, split, rep["name"], model) / "tuning.json"
    sig = signature(threshold, split, rep["name"], model)
    if path.exists():
        with open(path) as fh:
            record = json.load(fh)
        if record.get("signature") == sig:
            return record["params"], record["inner_cv_ROC_AUC"]
    _, _, train, _ = next(iter_folds(split))
    t = time.perf_counter()
    params, inner_auc = run_search(model, rep, split, threshold, train, SEED)
    record = {"signature": sig, "threshold": threshold, "split": split, "representation": rep["name"],
              "model": model, "tuning_fold": "repeat 0, fold 0", "n_train": int(len(train)), "seed": SEED,
              "params": to_builtin(params), "inner_cv_ROC_AUC": inner_auc,
              "search_seconds": round(time.perf_counter() - t, 1)}
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as fh:
        json.dump(record, fh, indent=1, default=str)
    log(f"    tuned [ee{threshold} | {split} | {rep['name']} | {model}] in {record['search_seconds'] / 60:.1f} min: "
        f"{ {k.replace('model__', ''): v for k, v in record['params'].items()} }")
    return record["params"], inner_auc


def fit_predict_classical(model, rep, split, threshold, train, test, seed):
    X, y = rep["X"], LABELS[threshold]
    est = make_estimator(model, rep, seed)
    if model in PARAM_SPACES:
        params, inner_auc = tuned_once(model, rep, split, threshold)
        est.set_params(**params)
        params = {k.replace("model__", ""): v for k, v in params.items()}
    else:
        params, inner_auc = {}, np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        est.fit(X[train], y[train])
    if model == "gp":
        params = est.fitted_params()
    score, label = scores_and_labels(model, est, X[test])
    return score, label, to_builtin(params), float(inner_auc)

## 6. Classical models

Each outer fold of each (threshold, split, representation, model) is stored in `cache/` as soon as it is
finished; a restarted run skips stored folds. The stored record carries a signature of the inputs (file hashes,
labels, fold file, search space, `N_ITER`, seeds, `CODE_VERSION`); records with a different signature are
recomputed. The search result is cached as `tuning.json` next to the fold records.

Loop order: threshold → representation → split → model, so that all results for the first threshold are
complete before the next one starts. Progress lines are appended, time-stamped and with the process memory,
to `run_log.txt`. Per-fold seed: `SEED + 1000·repeat + fold`.

In [ ]:
def signature(threshold, split, rep_name, model):
    payload = {"code": CODE_VERSION, "threshold": threshold, "labels": LABEL_HASHES[threshold],
               "ee_column": EE_COLUMN, "cutoff": DECISION_THRESHOLD, "split": split,
               "folds": FOLD_HASHES[split], "seed": SEED, "dataset": FILE_HASHES[DATASET_FILE],
               "conditions": FILE_HASHES[CONDITIONS_FILE], "model": model, "n_inner": N_INNER_FOLDS}
    if model == "chemprop":
        payload["chemprop"] = CHEMPROP
    else:
        payload["representation"] = FILE_HASHES.get(REPRESENTATIONS[rep_name][0])
        payload["space"] = PARAM_SPACES.get(model)
        payload["n_iter"] = N_ITER if model in PARAM_SPACES else None
        payload["gp_restarts"] = GP_N_RESTARTS if model == "gp" else None
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]


def combo_dir(threshold, split, rep_name, model):
    return CACHE_DIR / f"ee{threshold}" / split / rep_name / model


def load_cached(path, sig):
    if not path.exists():
        return None
    with open(path, "rb") as fh:
        record = pickle.load(fh)
    return record if record.get("signature") == sig else None


def save_cached(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as fh:
        pickle.dump(record, fh)
    os.replace(tmp, path)


def run_folds(threshold, split, rep_name, model, fit_fn):
    """Runs all outer folds of one combination with fold-level caching."""
    sig = signature(threshold, split, rep_name, model)
    y = LABELS[threshold]
    n_cached = n_fitted = 0
    t0 = time.perf_counter()
    for repeat, fold, train, test in iter_folds(split):
        path = combo_dir(threshold, split, rep_name, model) / f"r{repeat}_f{fold:03d}.pkl"
        if load_cached(path, sig) is not None:
            n_cached += 1
            continue
        seed = SEED + 1000 * repeat + fold
        t = time.perf_counter()
        score, label, params, inner_auc = fit_fn(split, threshold, train, test, seed)
        save_cached(path, {"signature": sig, "threshold": threshold, "split": split,
                           "representation": rep_name, "model": model, "repeat": repeat, "fold": fold,
                           "reaction_id": REACTION_IDS[test], "ee": EE[test], "y_true": y[test],
                           "score": np.asarray(score, dtype=float).ravel(),
                           "y_pred": np.asarray(label, dtype=int).ravel(),
                           "params": params, "inner_cv_ROC_AUC": inner_auc,
                           "fit_seconds": time.perf_counter() - t, "seed": seed})
        n_fitted += 1
    log(f"  [ee{threshold} | {split} | {rep_name} | {model}] folds: {n_cached} cached, {n_fitted} fitted "
        f"({(time.perf_counter() - t0) / 60:.1f} min) | memory {memory_mb():.0f} MB")

In [ ]:
CLASSICAL_MODELS = [m for m in MODEL_ORDER if m in RUN_MODELS and m != "chemprop"]

if CLASSICAL_MODELS:
    for threshold in RUN_THRESHOLDS:
        log(f"--- ee >= {threshold}% ---")
        for rep_name in RUN_REPRESENTATIONS:
            rep = load_representation(rep_name)
            if rep is None:
                log(f"{rep_name}: {REPRESENTATIONS[rep_name][0]} not found, skipped")
                continue
            n_total = rep["X"].shape[1]
            log(f"{rep_name}: {n_total} features ({rep['n_fp']} fingerprint, {n_total - rep['n_fp']} continuous)")
            for split in RUN_SPLITS:
                for model in CLASSICAL_MODELS:
                    fit_fn = (lambda s, t, tr, te, seed, m=model:
                              fit_predict_classical(m, rep, s, t, tr, te, seed))
                    run_folds(threshold, split, rep_name, model, fit_fn)
                release_memory()
            del rep, fit_fn
            release_memory()

## 7. chemprop D-MPNN

Same architecture and inputs as in the regression notebook, with a binary classification head
(sigmoid output, binary cross-entropy loss):
1. `mapped_rxn` as a condensed graph of reaction (CGR, mode `REAC_DIFF`);
2. `organocatalyst` as a molecular graph.

Extra descriptors: reaction conditions and `cat_axial_R` / `cat_axial_S`, imputed and standardised with
statistics of the training part of each fold. Class imbalance: per-reaction loss weights `n / (2·n_class)`
from the training part.

Hyperparameters are fixed. Early stopping and selection of the best epoch use the weighted validation loss
on an internal validation part of the training data (`val_fraction`): stratified by class for `random`,
by `research_group` for `lorgo_group`, by `catalyst_id` for `gkf10_catalyst`.

In [ ]:
CP = {}   # chemprop modules, featurizers and pre-parsed datapoints


def chemprop_setup():
    if CP:
        return CP
    from chemprop import data as cp_data, featurizers, models as cp_models, nn as cp_nn
    from lightning import pytorch as pl
    import torch

    for name in ("lightning", "lightning.pytorch", "lightning.fabric", "pytorch_lightning"):
        logging.getLogger(name).setLevel(logging.ERROR)
    warnings.filterwarnings("ignore", module="lightning")
    warnings.filterwarnings("ignore", message=".*num_workers.*")

    try:
        rxn_feat = featurizers.CondensedGraphOfReactionFeaturizer(mode_="REAC_DIFF")
    except TypeError:
        rxn_feat = featurizers.CondensedGraphOfReactionFeaturizer(mode="REAC_DIFF")
    if "REAC_DIFF" not in str(rxn_feat.mode).upper():
        raise RuntimeError(f"CGR mode is {rxn_feat.mode}, expected REAC_DIFF")
    mol_feat = featurizers.SimpleMoleculeMolGraphFeaturizer()

    rxn_dps, cat_dps, failed = [], [], []
    for rid, rxn, cat in zip(REACTION_IDS, data["mapped_rxn"], data["organocatalyst"]):
        try:
            rxn_dps.append(cp_data.ReactionDatapoint.from_smi(rxn))
            cat_dps.append(cp_data.MoleculeDatapoint.from_smi(cat))
        except Exception as exc:
            failed.append((rid, str(exc)[:80]))
    if failed:
        raise ValueError(f"{len(failed)} reactions could not be parsed by chemprop, e.g. {failed[:3]}")

    if TORCH_THREADS > 0:
        torch.set_num_threads(TORCH_THREADS)
    if CHEMPROP_DEVICE == "auto":
        accelerator = ("cuda" if torch.cuda.is_available() else
                       "mps" if torch.backends.mps.is_available() else "cpu")
    else:
        accelerator = CHEMPROP_DEVICE
    if accelerator == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS requested but not available")
    if accelerator == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but not available")
    if accelerator == "cuda":
        torch.set_float32_matmul_precision("high")

    class BestWeights(pl.Callback):
        """Keeps in memory the weights of the epoch with the lowest validation loss."""

        def __init__(self):
            self.score, self.epoch, self.state = float("inf"), -1, None

        def on_validation_end(self, trainer, module):
            if trainer.sanity_checking:
                return
            value = trainer.callback_metrics.get("val_loss")
            if value is not None and float(value) < self.score:
                self.score, self.epoch = float(value), int(trainer.current_epoch)
                self.state = {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

    CP.update(data=cp_data, models=cp_models, nn=cp_nn, pl=pl, BestWeights=BestWeights,
              rxn_feat=rxn_feat, mol_feat=mol_feat,
              rxn_dps=rxn_dps, cat_dps=cat_dps, accelerator=accelerator)
    device_name = torch.cuda.get_device_name(0) if accelerator == "cuda" else accelerator
    print(f"chemprop {VERSIONS['chemprop']} | accelerator: {device_name} | torch threads: {torch.get_num_threads()} | "
          f"{len(rxn_dps)} reactions parsed")
    return CP


def featurizer_dims(feat):
    try:
        return tuple(feat.shape)
    except AttributeError:
        return feat.atom_fdim, feat.bond_fdim


def chemprop_descriptors():
    """Conditions (merged on reaction_id) + axial flags, in dataset order."""
    return np.hstack([conditions[COND_COLS].to_numpy(float), AXIAL_FLAGS])


def fit_predict_chemprop(split, threshold, train, test, seed):
    cp = chemprop_setup()
    data_, nn_, pl = cp["data"], cp["nn"], cp["pl"]
    pl.seed_everything(seed, workers=True)
    y = LABELS[threshold]

    # internal validation part: stratified by class (random) or grouped like the outer split
    if split == "random":
        splitter, g = StratifiedShuffleSplit(1, test_size=CHEMPROP["val_fraction"], random_state=seed), None
        tr_i, va_i = next(splitter.split(np.zeros(len(train)), y[train]))
    else:
        splitter = GroupShuffleSplit(1, test_size=CHEMPROP["val_fraction"], random_state=seed)
        g = GROUPS[SPLIT_GROUP[split]][train]
        tr_i, va_i = next(splitter.split(np.zeros(len(train)), groups=g))
    tr, va = train[tr_i], train[va_i]

    XD = chemprop_descriptors()
    prep = BlockPreprocessor(0, scale_continuous=True).fit(XD[tr])
    n1 = y[tr].sum()
    class_weight = {0: len(tr) / (2 * (len(tr) - n1)), 1: len(tr) / (2 * n1)}

    def dataset(idx, with_y):
        xd = prep.transform(XD[idx]).astype(np.float32)
        rxn = [replace(cp["rxn_dps"][i], y=np.array([float(y[i])]) if with_y else None,
                       weight=class_weight[int(y[i])] if with_y else 1.0, x_d=xd[j])
               for j, i in enumerate(idx)]
        cat = [cp["cat_dps"][i] for i in idx]
        return data_.MulticomponentDataset([data_.ReactionDataset(rxn, cp["rxn_feat"]),
                                            data_.MoleculeDataset(cat, cp["mol_feat"])])

    c = CHEMPROP
    loaders = {
        "train": data_.build_dataloader(dataset(tr, True), batch_size=c["batch_size"], num_workers=0,
                                        seed=seed, shuffle=True),
        "val": data_.build_dataloader(dataset(va, True), batch_size=c["batch_size"], num_workers=0, shuffle=False),
        "test": data_.build_dataloader(dataset(test, False), batch_size=c["batch_size"], num_workers=0, shuffle=False),
    }

    blocks = [nn_.BondMessagePassing(*featurizer_dims(f), d_h=c["message_hidden_dim"], depth=c["depth"],
                                     dropout=c["dropout"]) for f in (cp["rxn_feat"], cp["mol_feat"])]
    mcmp = nn_.MulticomponentMessagePassing(blocks=blocks, n_components=2)
    ffn = nn_.BinaryClassificationFFN(input_dim=mcmp.output_dim + len(prep.cont_idx_),
                                      hidden_dim=c["ffn_hidden_dim"], n_layers=c["ffn_num_layers"],
                                      dropout=c["dropout"])
    # the logged auxiliary metric is the loss itself (AUROC is undefined on single-class validation parts)
    model = cp["models"].MulticomponentMPNN(mcmp, nn_.MeanAggregation(), ffn, batch_norm=False,
                                            metrics=[nn_.metrics.BCELoss()],
                                            warmup_epochs=c["warmup_epochs"], init_lr=c["init_lr"],
                                            max_lr=c["max_lr"], final_lr=c["final_lr"])

    best = cp["BestWeights"]()
    stop = pl.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=c["patience"])
    trainer = pl.Trainer(accelerator=cp["accelerator"], devices=1, max_epochs=c["max_epochs"],
                         callbacks=[best, stop], logger=False, enable_checkpointing=False,
                         enable_progress_bar=False, enable_model_summary=False)
    trainer.fit(model, loaders["train"], loaders["val"])
    if best.state is not None:
        model.load_state_dict(best.state)
    preds = trainer.predict(model, loaders["test"])
    prob = np.concatenate([p.detach().cpu().numpy().reshape(-1) for p in preds])   # sigmoid output
    params = {"best_epoch": best.epoch, "epochs_trained": int(trainer.current_epoch),
              "best_val_loss": best.score, "n_train": int(len(tr)), "n_val": int(len(va)),
              "val_class1_fraction": float(y[va].mean())}
    return prob, (prob >= DECISION_THRESHOLD).astype(int), params, np.nan

In [ ]:
if "chemprop" in RUN_MODELS:
    for threshold in RUN_THRESHOLDS:
        log(f"--- ee >= {threshold}% (chemprop) ---")
        for split in RUN_SPLITS:
            run_folds(threshold, split, "cgr", "chemprop", fit_predict_chemprop)
            gc.collect()

## 8. Collected results

All cached combinations with a valid signature and a complete set of outer folds are collected,
including those from earlier runs with other `RUN_*` selections (e.g. the chemprop run). Models listed in
`EXCLUDE_MODELS` are left out (their cached folds stay on disk).

In [ ]:
if not COLLECT:
    print("Skipped: worker run (MANNICH_COLLECT=0)")
else:
    records, stale = [], 0
    for path in sorted(CACHE_DIR.rglob("*.pkl")):
        with open(path, "rb") as fh:
            rec = pickle.load(fh)
        if rec["threshold"] not in EE_THRESHOLDS or rec["split"] not in ALL_SPLITS or rec["model"] in EXCLUDE_MODELS:
            continue
        if rec["representation"] != "cgr" and REPRESENTATIONS[rec["representation"]][0] not in FILE_HASHES:
            stale += 1
            continue
        if rec["signature"] != signature(rec["threshold"], rec["split"], rec["representation"], rec["model"]):
            stale += 1
            continue
        records.append(rec)

    by_combo = {}
    for rec in records:
        by_combo.setdefault((rec["threshold"], rec["split"], rec["representation"], rec["model"]), []).append(rec)

    complete, incomplete = {}, []
    for key, recs in by_combo.items():
        if len(recs) == n_fold_units(key[1]):
            complete[key] = recs
        else:
            incomplete.append((*key, len(recs), n_fold_units(key[1])))

    pred_rows, hp_rows = [], []
    for (threshold, split, rep_name, model), recs in complete.items():
        for rec in recs:
            pred_rows.append(pd.DataFrame({"reaction_id": rec["reaction_id"], "threshold": threshold, "split": split,
                                           "repeat": rec["repeat"], "fold": rec["fold"],
                                           "representation": rep_name, "model": model, "ee": rec["ee"],
                                           "y_true": rec["y_true"], "score": rec["score"], "y_pred": rec["y_pred"]}))
            hp_rows.append({"threshold": threshold, "split": split, "repeat": rec["repeat"], "fold": rec["fold"],
                            "representation": rep_name, "model": model, "seed": rec["seed"],
                            "n_test": len(rec["reaction_id"]), "inner_cv_ROC_AUC": rec["inner_cv_ROC_AUC"],
                            "fit_seconds": round(rec["fit_seconds"], 2),
                            "params": json.dumps(rec["params"], sort_keys=True, default=str)})

    if not pred_rows:
        raise RuntimeError("No complete combinations in the cache")

    SORT = ["threshold", "split", "representation", "model", "repeat", "fold"]
    PRED = pd.concat(pred_rows, ignore_index=True).sort_values(SORT + ["reaction_id"], ignore_index=True)
    HPARAMS = pd.DataFrame(hp_rows).sort_values(SORT, ignore_index=True)
    PRED.to_csv(RESULTS_DIR / "predictions_oof.csv", index=False)
    HPARAMS.to_csv(RESULTS_DIR / "hyperparameters_by_fold.csv", index=False)

    tuning_rows = []
    for path in sorted(CACHE_DIR.rglob("tuning.json")):
        with open(path) as fh:
            rec = json.load(fh)
        key = (rec["threshold"], rec["split"], rec["representation"], rec["model"])
        if key in complete and rec["signature"] == signature(*key):
            tuning_rows.append({k: rec[k] for k in ("threshold", "split", "representation", "model", "tuning_fold",
                                                    "n_train", "inner_cv_ROC_AUC", "search_seconds")}
                               | {"params": json.dumps({k.replace("model__", ""): v for k, v in rec["params"].items()},
                                                       sort_keys=True, default=str)})
    if tuning_rows:
        pd.DataFrame(tuning_rows).sort_values(["threshold", "split", "representation", "model"]).to_csv(
            RESULTS_DIR / "hyperparameters_tuned.csv", index=False)

    print(f"Complete combinations: {len(complete)} | stale fold records ignored: {stale}")
    if incomplete:
        print("Incomplete combinations (excluded):")
        for threshold, split, rep_name, model, n_have, n_need in sorted(incomplete):
            print(f"  ee{threshold} | {split} | {rep_name} | {model}: {n_have}/{n_need} folds")

## 9. Metrics

All metrics are computed on pooled out-of-fold predictions (all test folds of one partition together); for
`random`, per repeat and reported as mean ± sd over the 5 repeats.

| Metric | Uses | Chance level |
|---|---|---|
| ROC-AUC (primary) | score | 0.5 |
| PR-AUC (average precision) | score | prevalence of class 1 |
| MCC (Matthews correlation coefficient) | labels | 0 |
| F1 (class 1) | labels | 2p/(1 + p) for "always class 1", p = prevalence |
| Balanced accuracy | labels | 0.5 |

F1 refers to class 1 (high ee) and depends on its prevalence: when class 1 is the majority (low thresholds),
a classifier that always predicts class 1 reaches a high F1, so F1 is read together with MCC and balanced
accuracy. Pooling scores over folds compares scores of models trained on different folds; under
leave-one-group-out this is the only option, because many test folds hold one class only.

95% CIs: cluster bootstrap with `N_BOOT` replicates, as in the regression notebook (clusters: reactions for
`random`, research groups for `lorgo_group`, catalysts for `gkf10_catalyst`; for `random` the statistic of each
replicate is the mean over repeats). The same replicates are used for all thresholds, models and representations
within a split. The confusion-matrix counts (TP, FP, TN, FN) are given per partition (mean over repeats for
`random`).

In [ ]:
if not COLLECT:
    print("Skipped: worker run (MANNICH_COLLECT=0)")
else:
    CLUSTER_OF_SPLIT = {"random": None, "lorgo_group": "research_group", "gkf10_catalyst": "catalyst_id"}
    METRIC_NAMES = ["ROC_AUC", "PR_AUC", "MCC", "F1", "BA"]
    METRIC_LABELS = {"ROC_AUC": "ROC-AUC", "PR_AUC": "PR-AUC", "MCC": "MCC", "F1": "F1", "BA": "Balanced accuracy"}


    def bootstrap_weights(split, n_boot, seed):
        """(n_boot, n_reactions) multiplicities from resampling clusters with replacement."""
        rng = np.random.default_rng([seed, ALL_SPLITS.index(split)])
        col = CLUSTER_OF_SPLIT[split]
        codes = np.arange(len(REACTION_IDS)) if col is None else pd.factorize(GROUPS[col])[0]
        n_clusters = codes.max() + 1
        draws = rng.integers(0, n_clusters, size=(n_boot, n_clusters))
        counts = np.zeros((n_boot, n_clusters))
        np.add.at(counts, (np.repeat(np.arange(n_boot), n_clusters), draws.ravel()), 1)
        return counts[:, codes], n_clusters


    def weighted_metrics(W, y, score, label):
        """Classification metrics for each row of the weight matrix W (n_boot, n).

        ROC-AUC with half credit for tied scores and average precision (step-wise, as in scikit-learn),
        computed from cumulative class weights over score groups in descending order.
        """
        W = np.atleast_2d(W).astype(float)
        y = np.asarray(y, dtype=float)
        order = np.argsort(-score, kind="mergesort")
        s, yo, Wo = score[order], y[order], W[:, order]
        starts = np.r_[0, np.flatnonzero(np.diff(s) != 0) + 1]          # groups of tied scores
        P = np.add.reduceat(Wo * yo, starts, axis=1)
        N = np.add.reduceat(Wo * (1 - yo), starts, axis=1)
        TPc, FPc = np.cumsum(P, axis=1), np.cumsum(N, axis=1)
        wp, wn = TPc[:, -1], FPc[:, -1]
        with np.errstate(divide="ignore", invalid="ignore"):
            auc = (P * (wn[:, None] - FPc + 0.5 * N)).sum(1) / (wp * wn)
            ap = (P * TPc / np.where(TPc + FPc > 0, TPc + FPc, 1)).sum(1) / wp
        yl = np.asarray(label, dtype=float)
        tp, fp = W @ (y * yl), W @ ((1 - y) * yl)
        fn, tn = W @ (y * (1 - yl)), W @ ((1 - y) * (1 - yl))
        with np.errstate(divide="ignore", invalid="ignore"):
            denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
            mcc = np.where(denom > 0, (tp * tn - fp * fn) / np.where(denom > 0, denom, 1), 0.0)
            f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
            ba = 0.5 * (tp / (tp + fn) + tn / (tn + fp))
        return {"ROC_AUC": auc, "PR_AUC": ap, "MCC": mcc, "F1": f1, "BA": ba,
                "TP": tp, "FP": fp, "TN": tn, "FN": fn}


    # consistency of the vectorised metrics with scikit-learn on one combination
    _s = PRED.groupby(["threshold", "split", "representation", "model", "repeat"]).get_group(
        tuple(PRED.iloc[0][["threshold", "split", "representation", "model", "repeat"]]))
    _m = weighted_metrics(np.ones(len(_s)), _s["y_true"].to_numpy(), _s["score"].to_numpy(), _s["y_pred"].to_numpy())
    _ref = {"ROC_AUC": roc_auc_score(_s["y_true"], _s["score"]),
            "PR_AUC": average_precision_score(_s["y_true"], _s["score"]),
            "MCC": matthews_corrcoef(_s["y_true"], _s["y_pred"]),
            "F1": f1_score(_s["y_true"], _s["y_pred"], zero_division=0),
            "BA": balanced_accuracy_score(_s["y_true"], _s["y_pred"])}
    for k, v in _ref.items():
        if not np.isclose(_m[k][0], v, atol=1e-9):
            raise AssertionError(f"{k}: vectorised {_m[k][0]} vs scikit-learn {v}")

    BOOT = {}
    alpha = (1 - CI_LEVEL) / 2
    rows = []
    for (threshold, split, rep_name, model), sub in PRED.groupby(["threshold", "split", "representation", "model"],
                                                                 sort=False):
        if split not in BOOT:
            BOOT[split] = bootstrap_weights(split, N_BOOT, SEED)
        W, n_clusters = BOOT[split]
        count = sub.pivot_table(index="reaction_id", columns="repeat", values="score", aggfunc="count")
        if not (count == 1).all().all() or len(count) != len(REACTION_IDS):
            raise ValueError(f"ee{threshold}/{split}/{rep_name}/{model}: each reaction must be predicted once per repeat")
        score = sub.pivot(index="reaction_id", columns="repeat", values="score").loc[REACTION_IDS].to_numpy()
        label = sub.pivot(index="reaction_id", columns="repeat", values="y_pred").loc[REACTION_IDS].to_numpy()
        y = LABELS[threshold]
        n_rep = score.shape[1]
        point = {k: [] for k in METRIC_NAMES + ["TP", "FP", "TN", "FN"]}
        boot = {k: 0.0 for k in METRIC_NAMES}
        for r in range(n_rep):
            for k, v in weighted_metrics(np.ones(len(y)), y, score[:, r], label[:, r]).items():
                point[k].append(float(v[0]))
            for k, v in weighted_metrics(W, y, score[:, r], label[:, r]).items():
                if k in boot:
                    boot[k] = boot[k] + v / n_rep
        row = {"threshold": threshold, "representation": rep_name, "model": model, "split": split,
               "n_reactions": len(y), "prevalence": y.mean(), "n_repeats": n_rep,
               "n_folds": sub.groupby(["repeat", "fold"]).ngroups // n_rep, "n_clusters": n_clusters}
        for k in METRIC_NAMES:
            vals = np.asarray(point[k])
            row[k] = vals.mean()
            row[f"{k}_sd"] = vals.std(ddof=1) if n_rep > 1 else np.nan
            row[f"{k}_ci_low"], row[f"{k}_ci_high"] = np.nanquantile(boot[k], [alpha, 1 - alpha])
        for k in ("TP", "FP", "TN", "FN"):
            row[k] = float(np.mean(point[k]))
        rows.append(row)

    METRICS = pd.DataFrame(rows)
    METRICS["_t"] = METRICS["threshold"].map(EE_THRESHOLDS.index)
    METRICS["_s"] = METRICS["split"].map(ALL_SPLITS.index)
    METRICS["_r"] = METRICS["representation"].map(REP_ORDER.index)
    METRICS["_m"] = METRICS["model"].map(MODEL_ORDER.index)
    METRICS = (METRICS.sort_values(["_t", "_s", "_r", "_m"]).drop(columns=["_t", "_s", "_r", "_m"])
               .reset_index(drop=True))
    METRICS.to_csv(RESULTS_DIR / "metrics_summary.csv", index=False, float_format="%.4f")


    def fmt(row, k):
        sd = f" ± {row[f'{k}_sd']:.3f}" if pd.notna(row[f"{k}_sd"]) else ""
        return f"{row[k]:.3f}{sd} [{row[f'{k}_ci_low']:.3f}, {row[f'{k}_ci_high']:.3f}]"


    for threshold in [t for t in EE_THRESHOLDS if t in METRICS["threshold"].unique()]:
        for split in [s for s in ALL_SPLITS if s in METRICS["split"].unique()]:
            sub = METRICS[(METRICS["threshold"] == threshold) & (METRICS["split"] == split)]
            if sub.empty:
                continue
            table = pd.DataFrame({"Representation": sub["representation"].map(REP_LABELS),
                                  "Model": sub["model"].map(MODEL_LABELS)}
                                 | {METRIC_LABELS[k]: [fmt(r, k) for _, r in sub.iterrows()] for k in METRIC_NAMES})
            print(f"\nee ≥ {threshold}% (prevalence {sub['prevalence'].iloc[0]:.3f}) — {SPLIT_LABELS[split]} — "
                  f"value{' ± sd over repeats' if split == 'random' else ''} [{int(CI_LEVEL * 100)}% CI]")
            display(table.style.hide(axis="index"))

## 10. Figures

- `heatmap_roc_auc`, `heatmap_mcc`: representation × model, one panel per split (rows) and threshold
  (columns); diverging scale centred at the chance level (ROC-AUC 0.5, MCC 0), common to all panels.
- `roc_pr_ee<threshold>`: ROC and precision–recall curves, one row per split; for each model the representation
  with the highest ROC-AUC in that split. For `random`, curves are drawn from the predictions of all repeats
  pooled; legend values are the table values (mean over repeats). Dashed lines: chance level (ROC diagonal;
  prevalence for PR).

Each figure is written as PDF and PNG (300 dpi) to `figures/`.

In [ ]:
if not COLLECT:
    print("Skipped: worker run (MANNICH_COLLECT=0)")
else:
    TEXT_COLOR = "#222222"
    MODEL_COLORS = {"logreg": "#2a78d6", "svc": "#eb6834", "gp": "#1baf7a", "rf": "#eda100",
                    "xgb": "#e87ba4", "chemprop": "#008300"}       # fixed categorical order
    MODEL_STYLES = {"logreg": "-", "svc": "--", "gp": "-.", "rf": ":", "xgb": (0, (5, 1, 1, 1)), "chemprop": "-"}


    def save_figure(fig, name):
        for ext in ("pdf", "png"):
            fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")


    def present(values, order):
        return [x for x in order if x in set(values)]


    def annotate_cells(ax, M, im):
        for (i, j), v in np.ndenumerate(M):
            if np.isnan(v):
                continue
            r, g, b, _ = im.cmap(im.norm(v))
            luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if luminance < 0.5 else TEXT_COLOR)


    def metric_heatmap(metric, centre, name):
        thresholds = present(METRICS["threshold"], EE_THRESHOLDS)
        splits = present(METRICS["split"], ALL_SPLITS)
        reps = present(METRICS["representation"], REP_ORDER)
        models = present(METRICS["model"], MODEL_ORDER)
        half = max(0.05, float(np.nanmax(np.abs(METRICS[metric] - centre))))
        norm = TwoSlopeNorm(vcenter=centre, vmin=centre - half, vmax=centre + half)
        fig, axes = plt.subplots(len(splits), len(thresholds), squeeze=False, layout="constrained",
                                 figsize=(len(thresholds) * (0.5 * len(models) + 1.3),
                                          len(splits) * (0.28 * len(reps) + 0.9)))
        for i, split in enumerate(splits):
            for j, threshold in enumerate(thresholds):
                ax = axes[i, j]
                sub = METRICS[(METRICS["split"] == split) & (METRICS["threshold"] == threshold)]
                M = (sub.pivot(index="representation", columns="model", values=metric)
                        .reindex(index=reps, columns=models).to_numpy(float))
                im = ax.imshow(np.ma.masked_invalid(M), cmap="RdBu", norm=norm, aspect="auto")
                ax.set_facecolor("#f2f2f2")
                annotate_cells(ax, M, im)
                ax.set_xticks(range(len(models)), [MODEL_LABELS[m] for m in models], rotation=45, ha="right")
                ax.set_yticks(range(len(reps)), [REP_LABELS[r] for r in reps] if j == 0 else [])
                ax.tick_params(length=0)
                for side in ax.spines.values():
                    side.set_visible(False)
                ax.set_title(f"{SPLIT_LABELS[split]} — ee ≥ {threshold}%", fontsize=7.5)
        cb = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
        cb.set_label(METRIC_LABELS[metric])
        cb.ax.tick_params(labelsize=6.5)
        cb.outline.set_visible(False)
        save_figure(fig, name)
        plt.show()


    metric_heatmap("ROC_AUC", 0.5, "heatmap_roc_auc")
    metric_heatmap("MCC", 0.0, "heatmap_mcc")

In [ ]:
if not COLLECT:
    print("Skipped: worker run (MANNICH_COLLECT=0)")
else:
    for threshold in present(PRED["threshold"], EE_THRESHOLDS):
        splits = present(PRED.loc[PRED["threshold"] == threshold, "split"], ALL_SPLITS)
        fig, axes = plt.subplots(len(splits), 2, figsize=(6.4, 3.9 * len(splits)), squeeze=False, layout="constrained")
        for i, split in enumerate(splits):
            msub = METRICS[(METRICS["threshold"] == threshold) & (METRICS["split"] == split)]
            prevalence = LABELS[threshold].mean()
            ax_roc, ax_pr = axes[i]
            ax_roc.plot([0, 1], [0, 1], color="0.6", lw=0.8, ls="--")
            ax_pr.axhline(prevalence, color="0.6", lw=0.8, ls="--")
            for model in [m for m in MODEL_ORDER if m != "dummy" and m in set(msub["model"])]:
                best = msub[msub["model"] == model].sort_values("ROC_AUC", ascending=False).iloc[0]
                d = PRED[(PRED["threshold"] == threshold) & (PRED["split"] == split)
                         & (PRED["model"] == model) & (PRED["representation"] == best["representation"])]
                fpr, tpr, _ = roc_curve(d["y_true"], d["score"])
                prec, rec, _ = precision_recall_curve(d["y_true"], d["score"])
                style = dict(color=MODEL_COLORS[model], ls=MODEL_STYLES[model], lw=1.2)
                name = f"{MODEL_LABELS[model]} ({REP_LABELS[best['representation']]})"
                ax_roc.plot(fpr, tpr, label=f"{name}, {best['ROC_AUC']:.2f}", **style)
                ax_pr.plot(rec, prec, label=f"{name}, {best['PR_AUC']:.2f}", **style)
            ax_roc.set(xlim=(0, 1), ylim=(0, 1.01), xlabel="False positive rate", ylabel="True positive rate",
                       aspect="equal")
            ax_pr.set(xlim=(0, 1), ylim=(0, 1.01), xlabel="Recall", ylabel="Precision", aspect="equal")
            ax_roc.set_title(f"{SPLIT_LABELS[split]} — ROC", fontsize=8)
            ax_pr.set_title(f"{SPLIT_LABELS[split]} — precision–recall", fontsize=8)
            for ax, title in ((ax_roc, "ROC-AUC"), (ax_pr, "PR-AUC")):   # legends below the panels, off the curves
                ax.legend(title=title, fontsize=5.5, title_fontsize=6, frameon=False, loc="upper center",
                          bbox_to_anchor=(0.5, -0.2), ncol=2)
        fig.suptitle(f"ee ≥ {threshold}% (prevalence {LABELS[threshold].mean():.2f})", fontsize=9)
        save_figure(fig, f"roc_pr_ee{threshold}")
        plt.show()

## 11. Manifest

In [ ]:
if not COLLECT:
    print("Skipped: worker run (MANNICH_COLLECT=0)")
else:
    manifest = {
        "notebook": "mannich_classification.ipynb",
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "packages": {p: package_version(p) for p in PACKAGES},
        "seeds": {"SEED": SEED, "random_repeat_seeds": RANDOM_REPEAT_SEEDS,
                  "per_fold_seed": "SEED + 1000*repeat + fold", "bootstrap_seed": [SEED, "split index"]},
        "target": {"ee_column": EE_COLUMN, "thresholds": EE_THRESHOLDS, "class_1": "abs(ee) >= threshold",
                   "prevalence": {str(t): float(LABELS[t].mean()) for t in EE_THRESHOLDS},
                   "decision_threshold": DECISION_THRESHOLD},
        "config": {"EXCLUDE_MODELS": EXCLUDE_MODELS, "RUN_REPRESENTATIONS": RUN_REPRESENTATIONS, "RUN_MODELS": RUN_MODELS, "RUN_SPLITS": RUN_SPLITS,
                   "N_INNER_FOLDS": N_INNER_FOLDS, "N_ITER": N_ITER, "HP_MODE": "once", "N_BOOT": N_BOOT,
                   "CI_LEVEL": CI_LEVEL, "GP_N_RESTARTS": GP_N_RESTARTS, "CHEMPROP": CHEMPROP,
                   "CHEMPROP_DEVICE": CHEMPROP_DEVICE, "PARAM_SPACES": PARAM_SPACES, "CODE_VERSION": CODE_VERSION},
        "input_files_sha256": FILE_HASHES,
        "fold_files_sha256": {f"folds_{s}.csv": h for s, h in FOLD_HASHES.items()},
        "results": {"complete_combinations": sorted("|".join(map(str, k)) for k in complete),
                    "incomplete_combinations": [list(map(str, x)) for x in sorted(incomplete)]},
    }
    with open(RESULTS_DIR / "manifest.json", "w") as fh:
        json.dump(manifest, fh, indent=2, default=str)
    print(f"Written: {sorted(p.name for p in RESULTS_DIR.iterdir() if p.is_file())}")